In [1]:
import numpy as np
import pickle

In [2]:
import h5py
import argparse

In [3]:
parser = argparse.ArgumentParser(description='Generate Transition1x pickle data')
parser.add_argument('--hdf5_file')
parser.add_argument('--datasplit_folder')
args = parser.parse_args(['--hdf5_file', '/home/yufeiluo/research/dataset/Transition1x/transition1x.h5', 
                          '--datasplit_folder', '/home/yufeiluo/research/transition_state_pred/Data'])

In [4]:
REFERENCE_ENERGIES = {
    1: -13.62222753701504,
    6: -1029.4130839658328,
    7: -1484.8710358098756,
    8: -2041.8396277138045,
    9: -2712.8213146878606,
}


def get_molecular_reference_energy(atomic_numbers):
    molecular_reference_energy = 0
    for atomic_number in atomic_numbers:
        molecular_reference_energy += REFERENCE_ENERGIES[atomic_number]

    return molecular_reference_energy

def generator(formula, rxn, grp):
    """ Iterates through a h5 group """

    energies = grp["wB97x_6-31G(d).energy"]
    forces = grp["wB97x_6-31G(d).forces"]
    atomic_numbers = list(grp["atomic_numbers"])
    positions = grp["positions"]
    molecular_reference_energy = get_molecular_reference_energy(atomic_numbers)

    for energy, force, position in zip(energies, forces, positions):
        d = {
            "rxn": rxn,
            "wB97x_6-31G(d).energy": energy.__float__(),
            "wB97x_6-31G(d).atomization_energy": energy
            - molecular_reference_energy.__float__(),
            "wB97x_6-31G(d).forces": force.tolist(),
            "positions": position,
            "formula": formula,
            "charges": atomic_numbers,
            'num_atoms': len(atomic_numbers)
        }

        yield d

In [5]:
def generate_data(hdf5_file, datasplit_folder, datasplit):
    reactant_list = []
    product_list = []
    transition_state_list = []
    assert datasplit in [
        "train",
        "valid",
        "test",
    ]
    with open(datasplit_folder+'/reactions_'+datasplit+'.pickle', 'rb') as f:
        datalist = pickle.load(f)
    with h5py.File(hdf5_file, "r") as f:
        data = f['data']
        i = 0
        for formula, rxn in datalist:
            reactant = next(generator(formula, rxn, data[formula][rxn]["reactant"]))
            product = next(generator(formula, rxn, data[formula][rxn]["product"]))
            transition_state = next(generator(formula, rxn, data[formula][rxn]["transition_state"]))
            reactant_list.append(reactant)
            product_list.append(product)
            transition_state_list.append(transition_state)

    res = {'reactant': {'ediff': []}, 'product': {'ediff': []}, 'transition_state': {}}
    for key in reactant_list[0].keys():
        for key_res in res:
            res[key_res][key] = []
    for i in range(len(reactant_list)):
        for key in reactant_list[i].keys():
            res['reactant'][key].append(reactant_list[i][key])
            res['product'][key].append(product_list[i][key])
            res['transition_state'][key].append(transition_state_list[i][key])
        res['reactant']['ediff'].append(reactant_list[i]["wB97x_6-31G(d).energy"] - transition_state_list[i]["wB97x_6-31G(d).energy"])
        res['product']['ediff'].append(product_list[i]["wB97x_6-31G(d).energy"] - transition_state_list[i]["wB97x_6-31G(d).energy"])

    return res


In [6]:
train_data = generate_data(args.hdf5_file, args.datasplit_folder, 'train')
valid_data = generate_data(args.hdf5_file, args.datasplit_folder, 'valid')
test_data = generate_data(args.hdf5_file, args.datasplit_folder, 'test')
with open('train.pkl', 'wb') as f:
    pickle.dump(train_data, f)
with open('valid.pkl', 'wb') as f:
    pickle.dump(valid_data, f)
with open('test.pkl', 'wb') as f:
    pickle.dump(test_data, f)